In [ ]:
import numpy as np
import open3d as o3d
from scipy.integrate import solve_ivp
from math import comb

# ---------- 1. 重新求解径向 ODE（稳定分支） ----------
n, k = 3, 2
R, r_max = 1.0, 10.0

def ode_system(r, y):
    u, v = y
    v_safe = max(v, 1e-5)
    dvdr = -((comb(n,k-1)/comb(n,k)) * (v_safe/r) * (1 - v_safe**(2*k)))
    return [v, dvdr]

sol = solve_ivp(ode_system, [R, r_max], [0.0, 0.99], 
                method='BDF', rtol=1e-8, atol=1e-8, max_step=0.05)

r_vals = sol.t
u_vals = sol.y[0]
v_vals = sol.y[1]

# ---------- 2. 生成 3D 旋转曲面点云 ----------
theta = np.linspace(0, 2*np.pi, 60)  # 60 个角度
points = []
for i in range(len(r_vals)):
    x = r_vals[i]
    y = u_vals[i]
    z = v_vals[i]
    # 绕 x 轴旋转：把 (x, y, z) 当作母线上的点，旋转生成圆环
    # 这里简化：把 (r, u(r), 0) 旋转，得到曲面上的点
    for th in theta:
        px = x
        py = y * np.cos(th)
        pz = y * np.sin(th)
        points.append([px, py, pz])

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(np.array(points))

# ---------- 3. 渲染 ----------
o3d.visualization.draw_geometries([pcd], window_name="Radial ODE Surface",
                                  width=800, height=600)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
o3d.io.write_point_cloud("radial_ode_surface.ply", pcd)

True